In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from IPython.display import Latex, HTML, Math, display
from uncertainties import ufloat
from uncertainties.umath import sqrt
from uncertainties import unumpy as unp
from scipy.stats import linregress
from scipy.optimize import curve_fit
from uncertainties.umath import sin, radians 
from uncertainties.umath import *
from scipy.signal import savgol_filter, find_peaks

In [ ]:
# 1. Franck-Hertz-Versuch

#daten aus csv laden
data = pd.read_csv("", sep=",", decimal=".", engine="python")           #csv file aus CASSY/OceanView?
I_A = data[""].to_numpy()                   #[A] Strom I_A; umrechenen aus der gemessenen Spannung!
U2 = data[""].to_numpy()                    #[V] Spannung U2

I_A_smooth = savgol_filter(I_A, window_length=21, polyorder=3)          #[A]; daten glätten um min und max zu finden 

#maxima und minima
peaks, _ = find_peaks(I_A_smooth, distance=20, prominence=0.05)         #indizes der maxima

valleys, _ = find_peaks(-I_A_smooth, distance=20, prominence=0.05)      #indizes der minima

#Plot

plt.figure()
plt.plot(U2, I_A, color="grey", alpha=0.3, label="Rohdaten")
plt.plot(U2, I_A_smooth, color="blue", label="geglättete Daten")
plt.plot(U2[peaks], I_A_smooth[peaks], 'ro', label="Maxima")
for i, v in enumerate(U2[peaks]):
    plt.annotate(f"{v:.2f}V", (U2[peaks][i], I_A_smooth[peaks][i]),
                 textcoords="offset points", ytext=(0, 10), ha="center", color="red")
plt.plot(U2[valleys], I_A_smooth[valleys], 'gs', label="Maxima")
for i, v in enumerate(U2[peaks]):
    plt.annotate(f"{v:.2f}V", (U2[valleys][i], I_A_smooth[valleys][i]),
                 textcoords="offset points", ytext=(0, -15), ha="center", color="green")
plt.xlabel("Beschleunigungsspannung U_2 [V]")
plt.ylabel("Auffangstrom I_A [A]")
plt.grid(True)
plt.legend()
plt.show()

#ergebnisse
print(f"Maxima gefunden bei: {U2[peaks]} [V]")
print(f"Minima gefunden bei: {U2[valleys]} [V]")
print()

peak_diffs = np.diff(U2[peaks])                 #Abstände zwischen Maxima [V]
valley_diffs = np.diff(U2[valleys])             #Abstände zwischen Minima [V]

combined_avg = np.mean(np.concatenate([peak_diffs, valley_diffs]))  #Mittelwert zwischen allen Abständen [V] (*e = eV)

print(f"Mittelwert Abstände Maxima: {np.mean(peak_diffs)} [V]")
print(f"Mittelwert Abstände Minima: {np.mean(valley_diffs)} [V]")
print(f"Abschätzung der Anregungsenergien: {combined_avg} [eV]")        #gesamter Mittelwert mit e multipliziert


#Analyse Leuchtzonen
U_leuchtzonen = np.array([])            #[V]

deltas = np.diff(U_leuchtzonen)
for i, delta in enumerate(deltas):
    print(f"Unterschied zwischen Zone {i+1} und {i+2}: {delta:.3f} [V]")

avg_delta = np.mean(deltas)

print()
print(f"Mittlerer Unterschied: {avg_delta:.3f} [V] = Anregungsenergie [eV]")

#Korrelation: 
#zw. 18 und 20: 3p in Ne ~18.7eV --> Fall 3p -> 3s



In [ ]:
# 2. Elektronenbeugung

#Konstanten
h = 6.626 * 10**(-34)                #Plank'sches Wirkungsquantum [Js]
e = 1.602 * 10**(-19)                #[C] Elementarladung
me = 9.109 * 10**(-31)               #[kg]  Elektronenmasse
L =                                  #[m] Abstand Graphit - Schirm

#Daten
U = np.array([])            #[V]

D1 = np.array([])           #[m], Durchmesser innerer Ring
D2 = np.array([])           #[m], Durchmesser äußerer Ring

#Berechnungen
lambdas = h / np.sqrt(2 * me * e * U)           #[m], de Broglie Wellenlänge
lambdas_plot = lambdas * 10**12                 #[m --> pm]

x1 = D1 / (2 * L)           #x-Achsen für D1, dimensionslos
x2 = D2 / (2 * L)           #x-Achsen für D2, dimensionslos

reg1 = linregress(x1, lambdas)
reg2 = linregress(x2, lambdas)

d1 = ufloat(reg1.slope, reg1.stderr) * 10**12       #[m --> pm]
d2 = ufloat(reg2.slope, reg2.stderr) * 10**12       #[m --> pm]

#Plot
plt.figure()
plt.plot(x1, lambdas_plot, 'o', color="blue", label=f"innerer Ring (d1 = {d1:.2f} pm)")
plt.plot(x1, reg1.slope*x1 + reg1.intercept, 'b--')
plt.plot(x2, lambdas_plot, 'o', color="red", label=f"innerer Ring (d2 = {d2:.2f} pm)")
plt.plot(x2, reg2.slope*x2 + reg2.intercept, 'r--')
plt.xlabel("D / 2L")
plt.ylabel("Wellenlänge λ [pm]")
plt.legend()
plt.grid(True)
plt.show()

print(f"d1: {d1} [pm]")
print(f"d2: {d2} [pm]")


